# Cleaning Data Indonesia OpenTender 2022 & 2023 & 2024
Output: `ocid | buyer_name | vendor_name | HPS | contract_value | procurement_method | category | item_description | date`

In [1]:
import pandas as pd


In [ ]:
FILES = [
    'datasets/indonesia_opentender_2024.xlsx',
    'datasets/indonesia_opentender_2023.xlsx',
    'datasets/indonesia_opentender_2022.xlsx',
]
def load_and_clean(filepath):
    xl = pd.ExcelFile(filepath)

    # --- Load sheets ---
    main = xl.parse('main')
    awards = xl.parse('awards')
    awards_suppliers = xl.parse('awards_suppliers')
    tender_items = xl.parse('tender_items')

    # --- Deduplicate item_description per ocid (ambil yang pertama) ---
    tender_items_dedup = (
        tender_items
        .dropna(subset=['main_ocid'])
        .groupby('main_ocid', as_index=False)
        .agg(item_description=('classification_description', lambda x: ' | '.join(x.dropna().unique())))
    )

    # --- Deduplicate vendor_name per ocid (gabungkan jika lebih dari satu) ---
    suppliers_dedup = (
        awards_suppliers
        .dropna(subset=['main_ocid'])
        .groupby('main_ocid', as_index=False)
        .agg(vendor_name=('name', lambda x: ' | '.join(x.dropna().unique())))
    )

    # --- Ambil contract_value dari awards (sum jika ada lebih dari satu award per ocid) ---
    awards_agg = (
        awards
        .dropna(subset=['main_ocid'])
        .groupby('main_ocid', as_index=False)
        .agg(contract_value=('value_amount', 'sum'))
    )

    # --- Pilih kolom dari main ---
    main_sel = main[[
        'ocid',
        'buyer_name',
        'tender_minValue_amount',
        'tender_additionalProcurementCategories',
        'tender_mainProcurementCategory',
        'date',
    ]].copy()

    main_sel.rename(columns={
        'tender_minValue_amount': 'HPS',
        'tender_additionalProcurementCategories': 'procurement_method',
        'tender_mainProcurementCategory': 'category',
    }, inplace=True)

    # --- Merge semua ---
    df = main_sel.merge(suppliers_dedup, left_on='ocid', right_on='main_ocid', how='left')
    df = df.merge(awards_agg, left_on='ocid', right_on='main_ocid', how='left')
    df = df.merge(tender_items_dedup, left_on='ocid', right_on='main_ocid', how='left')

    # --- Bersihkan kolom duplikat dari merge ---
    df.drop(columns=[c for c in df.columns if c.startswith('main_ocid')], inplace=True)

    # --- Parse tanggal ---
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.date

    # --- Urutan kolom final ---
    df = df[[
        'ocid', 'buyer_name', 'vendor_name',
        'HPS', 'contract_value',
        'procurement_method', 'category',
        'item_description', 'date'
    ]]

    # --- Reset index ---
    df.reset_index(drop=True, inplace=True)

    return df

In [3]:
# Proses setiap file dan tampilkan preview
dfs = {}
for f in FILES:
    print(f'\n>>> {f}')
    df = load_and_clean(f)
    dfs[f] = df
    print(f'Shape: {df.shape}')
    display(df.head(3))


>>> datasets/indonesia_opentender_2024.xlsx
Shape: (2629, 9)


,ocid,buyer_name,vendor_name,HPS,contract_value,procurement_method,category,item_description,date
0,ocds-20h3g7-12482010,Pemerintah Daerah Kota Surabaya,PT. DUTA BHUANA JAYA,7.445028e+08,6.666604e+08,consultancyServices,services,Jasa Konsultansi Badan Usaha Konstruksi,2024-02-02
1,ocds-20h3g7-12487010,Pemerintah Daerah Kota Surabaya,"MARGA PERKASA,CV",4.793422e+09,4.129909e+09,NaN,works,Pekerjaan Konstruksi,2024-01-15
2,ocds-20h3g7-12492010,Pemerintah Daerah Kota Surabaya,CV. Naga Kencana Wiratama,2.255991e+09,2.012700e+09,NaN,works,Pekerjaan Konstruksi,2024-01-15



>>> datasets/indonesia_opentender_2023.xlsx
Shape: (133846, 9)


,ocid,buyer_name,vendor_name,HPS,contract_value,procurement_method,category,item_description,date
0,ocds-20h3g7-10000065,Kementerian Agraria dan Tata Ruang/BPN,PT. ANDALAN REREKA CONSULTINDO,1.468459e+09,1.401695e+09,consultancyServices,services,Jasa Konsultansi Badan Usaha Non Konstruksi,2023-07-20
1,ocds-20h3g7-10002065,Kementerian Agraria dan Tata Ruang/BPN,Manokwari Maju,8.087027e+08,6.469622e+08,NaN,works,Pekerjaan Konstruksi,2023-07-10
2,ocds-20h3g7-10004107,Pemerintah Daerah Kabupaten Muara Enim,cv faulia jaya abadi,4.998500e+08,4.798930e+08,NaN,works,Pekerjaan Konstruksi,2023-11-11



>>> datasets/indonesia_opentender_2022.xlsx
Shape: (156377, 9)


,ocid,buyer_name,vendor_name,HPS,contract_value,procurement_method,category,item_description,date
0,ocds-20h3g7-10001054,Pemerintah Daerah Kabupaten Sleman,CV.JAYA MANUNGGAL,4.622517e+08,3.419078e+08,NaN,goods,Pengadaan Barang,2022-06-07
1,ocds-20h3g7-10003109,Kementerian Energi Dan Sumber Daya Mineral,INTI MITRA NIAGA,2.496739e+09,1.986794e+09,NaN,works,Pekerjaan Konstruksi,2022-08-21
2,ocds-20h3g7-10005423,Pemerintah Daerah Kabupaten Demak,CV. ATHREYA WISATA MAKMUR,2.999801e+08,2.971602e+08,NaN,services,Jasa Lainnya,2022-11-08


In [4]:
# Gabungkan kedua tahun menjadi satu dataframe
df_all = pd.concat(dfs.values(), ignore_index=True)

# Hapus duplikat berdasarkan ocid (jika ada)
df_all.drop_duplicates(subset='ocid', keep='first', inplace=True)
df_all.reset_index(drop=True, inplace=True)

print(f'Total baris gabungan: {len(df_all)}')
print(f'Missing values:')
display(df_all.isnull().sum())
display(df_all.head(5))

Total baris gabungan: 292852
Missing values:


ocid                       0
buyer_name              4661
vendor_name           128739
HPS                   129114
contract_value        128739
procurement_method    266524
category              128739
item_description      128739
date                       0
dtype: int64

,ocid,buyer_name,vendor_name,HPS,contract_value,procurement_method,category,item_description,date
0,ocds-20h3g7-12482010,Pemerintah Daerah Kota Surabaya,PT. DUTA BHUANA JAYA,7.445028e+08,6.666604e+08,consultancyServices,services,Jasa Konsultansi Badan Usaha Konstruksi,2024-02-02
1,ocds-20h3g7-12487010,Pemerintah Daerah Kota Surabaya,"MARGA PERKASA,CV",4.793422e+09,4.129909e+09,NaN,works,Pekerjaan Konstruksi,2024-01-15
2,ocds-20h3g7-12492010,Pemerintah Daerah Kota Surabaya,CV. Naga Kencana Wiratama,2.255991e+09,2.012700e+09,NaN,works,Pekerjaan Konstruksi,2024-01-15
3,ocds-20h3g7-12496010,Pemerintah Daerah Kota Surabaya,CV. Tiga Points Jaya Karya,3.425503e+09,2.566207e+09,NaN,works,Pekerjaan Konstruksi,2024-01-16
4,ocds-20h3g7-12504010,Pemerintah Daerah Kota Surabaya,CV. Citra Karya,1.307263e+09,1.231890e+09,NaN,works,Pekerjaan Konstruksi,2024-01-15


In [6]:
import os

DATA_DIR = 'datasets'

df_all.to_excel(os.path.join(DATA_DIR, 'opentender_clean_2022_2024.xlsx'), index=False)
df_all.to_csv(os.path.join(DATA_DIR, 'opentender_clean_2022_2024.csv'), index=False)
print('File tersimpan: opentender_clean_2022_2024.xlsx & .csv')

File tersimpan: opentender_clean_2022_2024.xlsx & .csv


In [2]:
df_opentender = pd.read_csv('datasets/opentender_clean_2022_2024.csv')
df_opentender.head()

,ocid,buyer_name,vendor_name,HPS,contract_value,procurement_method,category,item_description,date
0,ocds-20h3g7-12482010,Pemerintah Daerah Kota Surabaya,PT. DUTA BHUANA JAYA,7.445028e+08,6.666604e+08,consultancyServices,services,Jasa Konsultansi Badan Usaha Konstruksi,2024-02-02
1,ocds-20h3g7-12487010,Pemerintah Daerah Kota Surabaya,"MARGA PERKASA,CV",4.793422e+09,4.129909e+09,NaN,works,Pekerjaan Konstruksi,2024-01-15
2,ocds-20h3g7-12492010,Pemerintah Daerah Kota Surabaya,CV. Naga Kencana Wiratama,2.255991e+09,2.012700e+09,NaN,works,Pekerjaan Konstruksi,2024-01-15
3,ocds-20h3g7-12496010,Pemerintah Daerah Kota Surabaya,CV. Tiga Points Jaya Karya,3.425503e+09,2.566207e+09,NaN,works,Pekerjaan Konstruksi,2024-01-16
4,ocds-20h3g7-12504010,Pemerintah Daerah Kota Surabaya,CV. Citra Karya,1.307263e+09,1.231890e+09,NaN,works,Pekerjaan Konstruksi,2024-01-15
